# Chapter 12 - SVM Regression

In classification, the SVM tries to find the widest possible *street* (margin) between
classes while limiting margin violations. **Support Vector Regression (SVR)** flips the
idea: instead of keeping points *outside* the margin, the goal is to fit as many points
as possible *inside* a margin (the **epsilon-insensitive tube**) around the prediction.

**Topics covered:**
- The epsilon-insensitive tube: geometric intuition
- SVR in scikit-learn
- The epsilon parameter: width of the tube
- Kernel SVR: RBF, polynomial
- Comparing SVR to linear regression on nonlinear data
- Tuning SVR: C, epsilon, kernel, gamma
- Visualising the epsilon tube around predictions
- Pros and cons of SVM
- Summary: when to use SVM vs other methods

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from sklearn.svm import SVR
from sklearn.linear_model import LinearRegression
from sklearn.datasets import make_regression
from sklearn.model_selection import GridSearchCV, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

np.random.seed(42)
%matplotlib inline

---
## 1. The Epsilon-Insensitive Tube

In ordinary least squares, every residual contributes to the loss. SVR introduces an
**epsilon** ($\varepsilon$) parameter that defines a tube around the predicted function:

- Points **inside** the tube (residual $\leq \varepsilon$) incur **zero loss**.
- Points **outside** the tube are penalised linearly by how far they exceed $\varepsilon$.

Mathematically, the loss for a single point is:

$$L_\varepsilon(y, \hat{y}) = \max(0,\; |y - \hat{y}| - \varepsilon)$$

The optimisation problem then becomes:

$$\min_{\mathbf{w}, b} \; \frac{1}{2}\|\mathbf{w}\|^2 + C \sum_{i=1}^{n} L_\varepsilon(y_i, \hat{y}_i)$$

Just like in classification:
- **C** trades off model complexity (flatness of $\mathbf{w}$) against tolerance for
  points outside the tube.
- **Support vectors** are the points that lie *on or outside* the tube boundary — only
  they influence the fitted function.

In [ ]:
# Generate simple nonlinear data
np.random.seed(42)
X_demo = np.sort(np.random.uniform(0, 5, 60)).reshape(-1, 1)
y_demo = np.sin(X_demo).ravel() + np.random.normal(0, 0.15, X_demo.shape[0])

# Fit SVR with epsilon tube
svr_demo = SVR(kernel='rbf', C=10, epsilon=0.2, gamma='scale')
svr_demo.fit(X_demo, y_demo)

# Dense prediction line
X_plot = np.linspace(0, 5, 300).reshape(-1, 1)
y_pred = svr_demo.predict(X_plot)

fig, ax = plt.subplots(figsize=(10, 6))

# Epsilon tube
ax.fill_between(X_plot.ravel(), y_pred - 0.2, y_pred + 0.2,
                alpha=0.2, color='green', label=f'$\\varepsilon$-tube ($\\varepsilon$=0.2)')
ax.plot(X_plot, y_pred, 'r-', linewidth=2, label='SVR prediction')
ax.scatter(X_demo, y_demo, c='steelblue', edgecolors='k', s=50, zorder=3, label='Data')

# Highlight support vectors
sv_idx = svr_demo.support_
ax.scatter(X_demo[sv_idx], y_demo[sv_idx], s=150, facecolors='none',
           edgecolors='gold', linewidths=2, zorder=4,
           label=f'Support vectors ({len(sv_idx)})')

ax.set_xlabel('x')
ax.set_ylabel('y')
ax.set_title('SVR: the epsilon-insensitive tube')
ax.legend(loc='best')
plt.tight_layout()
plt.show()

Points inside the green band contribute nothing to the loss. Only the gold-circled
support vectors (on or outside the tube) shape the regression curve.

---
## 2. SVR in Scikit-Learn

The API mirrors `SVC`:

```python
from sklearn.svm import SVR

svr = SVR(kernel='rbf', C=1.0, epsilon=0.1, gamma='scale')
svr.fit(X_train, y_train)
y_pred = svr.predict(X_test)
```

Key parameters:
- `kernel`: `'linear'`, `'poly'`, `'rbf'` (default), `'sigmoid'`
- `C`: regularisation (same trade-off as classification)
- `epsilon`: half-width of the insensitive tube
- `gamma`: kernel coefficient for RBF/poly/sigmoid

In [ ]:
# End-to-end example with make_regression
X_reg, y_reg = make_regression(n_samples=200, n_features=1, noise=15, random_state=42)

X_tr, X_te, y_tr, y_te = train_test_split(X_reg, y_reg, test_size=0.25, random_state=42)

pipe_svr = Pipeline([
    ('scaler', StandardScaler()),
    ('svr', SVR(kernel='rbf', C=100, epsilon=0.1, gamma='scale'))
])

pipe_svr.fit(X_tr, y_tr)
y_pred_te = pipe_svr.predict(X_te)

print(f"R2 score:  {r2_score(y_te, y_pred_te):.4f}")
print(f"RMSE:      {np.sqrt(mean_squared_error(y_te, y_pred_te)):.4f}")
print(f"MAE:       {mean_absolute_error(y_te, y_pred_te):.4f}")

In [ ]:
# Plot the regression fit
X_line = np.linspace(X_reg.min(), X_reg.max(), 300).reshape(-1, 1)
y_line = pipe_svr.predict(X_line)

fig, ax = plt.subplots(figsize=(9, 6))
ax.scatter(X_te, y_te, c='steelblue', edgecolors='k', s=40, label='Test data')
ax.scatter(X_tr, y_tr, c='lightgrey', edgecolors='k', s=30, alpha=0.5, label='Train data')
ax.plot(X_line, y_line, 'r-', linewidth=2, label='SVR prediction')
ax.set_xlabel('x')
ax.set_ylabel('y')
ax.set_title('SVR on make_regression data')
ax.legend()
plt.tight_layout()
plt.show()

---
## 3. The Epsilon Parameter

Epsilon controls the **width of the tube** (the "no-penalty" zone):

| Epsilon | Tube width | Support vectors | Model behaviour |
|---------|-----------|----------------|-----------------|
| **Small** (e.g. 0.01) | Narrow | More | Model is forced to fit closely — risk of overfitting |
| **Large** (e.g. 1.0) | Wide | Fewer | Model tolerates large residuals — risk of underfitting |

The sweet spot depends on the noise level in your data. If you expect residuals of
roughly $\pm 0.2$, setting $\varepsilon \approx 0.2$ is a reasonable starting point.

In [ ]:
eps_values = [0.01, 0.1, 0.5, 1.0, 2.0]

fig, axes = plt.subplots(1, 5, figsize=(25, 4.5))

for ax, eps in zip(axes, eps_values):
    svr_eps = SVR(kernel='rbf', C=10, epsilon=eps, gamma='scale')
    svr_eps.fit(X_demo, y_demo)
    y_eps = svr_eps.predict(X_plot)

    ax.fill_between(X_plot.ravel(), y_eps - eps, y_eps + eps,
                    alpha=0.2, color='green')
    ax.plot(X_plot, y_eps, 'r-', linewidth=2)
    ax.scatter(X_demo, y_demo, c='steelblue', edgecolors='k', s=40)
    ax.scatter(X_demo[svr_eps.support_], y_demo[svr_eps.support_],
               s=120, facecolors='none', edgecolors='gold', linewidths=2)
    ax.set_title(f'$\\varepsilon$ = {eps}\nSVs: {len(svr_eps.support_)}')
    ax.set_xlabel('x')
    ax.set_ylabel('y')
    ax.set_ylim(-1.8, 1.8)

plt.suptitle('Effect of epsilon on the SVR tube width', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

---
## 4. Kernel SVR: RBF and Polynomial

Just like SVC, SVR supports kernels. The kernel determines the shape of the regression
function:

- **linear**: fits a straight line (or hyperplane).
- **poly**: fits polynomial curves; the `degree` parameter controls complexity.
- **rbf**: fits smooth, flexible curves; `gamma` controls locality.

In [ ]:
# Generate more complex nonlinear data
np.random.seed(42)
X_nl = np.sort(np.random.uniform(0, 2 * np.pi, 80)).reshape(-1, 1)
y_nl = np.sin(X_nl).ravel() + 0.5 * np.cos(2 * X_nl).ravel() + np.random.normal(0, 0.2, len(X_nl))

X_nl_plot = np.linspace(0, 2 * np.pi, 300).reshape(-1, 1)

kernel_configs = [
    ('linear', SVR(kernel='linear', C=10, epsilon=0.1)),
    ('poly (d=2)', SVR(kernel='poly', degree=2, C=10, epsilon=0.1)),
    ('poly (d=4)', SVR(kernel='poly', degree=4, C=10, epsilon=0.1)),
    ('rbf', SVR(kernel='rbf', C=10, epsilon=0.1, gamma='scale')),
]

fig, axes = plt.subplots(1, 4, figsize=(22, 5))

for ax, (name, svr) in zip(axes, kernel_configs):
    svr.fit(X_nl, y_nl)
    y_svr = svr.predict(X_nl_plot)

    ax.scatter(X_nl, y_nl, c='steelblue', edgecolors='k', s=40)
    ax.plot(X_nl_plot, y_svr, 'r-', linewidth=2)
    ax.fill_between(X_nl_plot.ravel(), y_svr - 0.1, y_svr + 0.1,
                    alpha=0.15, color='green')
    ax.scatter(X_nl[svr.support_], y_nl[svr.support_],
               s=120, facecolors='none', edgecolors='gold', linewidths=2)
    r2 = r2_score(y_nl, svr.predict(X_nl))
    ax.set_title(f'{name}\n$R^2$ = {r2:.3f}, SVs = {len(svr.support_)}')
    ax.set_xlabel('x')
    ax.set_ylabel('y')

plt.suptitle('Kernel SVR comparison on nonlinear data', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

The linear kernel can only capture the overall trend. Polynomial kernels add curvature
but may oscillate at the boundaries. The RBF kernel typically gives the most flexible
and smooth fit.

---
## 5. Comparing SVR to Linear Regression on Nonlinear Data

Linear regression minimises the sum of squared residuals across *all* points. SVR with
an RBF kernel can capture nonlinear patterns that linear regression completely misses.
Let us compare both on the same dataset.

In [ ]:
# Split the nonlinear data
X_tr_nl, X_te_nl, y_tr_nl, y_te_nl = train_test_split(
    X_nl, y_nl, test_size=0.25, random_state=42
)

# Linear regression
lr = LinearRegression()
lr.fit(X_tr_nl, y_tr_nl)

# SVR with RBF kernel
svr_rbf = Pipeline([
    ('scaler', StandardScaler()),
    ('svr', SVR(kernel='rbf', C=100, epsilon=0.1, gamma='scale'))
])
svr_rbf.fit(X_tr_nl, y_tr_nl)

print("=== Linear Regression ===")
print(f"  Train R2: {lr.score(X_tr_nl, y_tr_nl):.4f}")
print(f"  Test  R2: {lr.score(X_te_nl, y_te_nl):.4f}")
print(f"  Test RMSE: {np.sqrt(mean_squared_error(y_te_nl, lr.predict(X_te_nl))):.4f}")

print("\n=== SVR (RBF) ===")
print(f"  Train R2: {r2_score(y_tr_nl, svr_rbf.predict(X_tr_nl)):.4f}")
print(f"  Test  R2: {r2_score(y_te_nl, svr_rbf.predict(X_te_nl)):.4f}")
print(f"  Test RMSE: {np.sqrt(mean_squared_error(y_te_nl, svr_rbf.predict(X_te_nl))):.4f}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, model, name in zip(axes, [lr, svr_rbf], ['Linear Regression', 'SVR (RBF)']):
    y_line = model.predict(X_nl_plot)
    ax.scatter(X_tr_nl, y_tr_nl, c='steelblue', edgecolors='k', s=50, label='Train')
    ax.scatter(X_te_nl, y_te_nl, c='orange', edgecolors='k', s=50, label='Test')
    ax.plot(X_nl_plot, y_line, 'r-', linewidth=2, label='Prediction')
    test_r2 = r2_score(y_te_nl, model.predict(X_te_nl))
    ax.set_title(f'{name}\nTest $R^2$ = {test_r2:.3f}')
    ax.set_xlabel('x')
    ax.set_ylabel('y')
    ax.legend()

plt.suptitle('Linear Regression vs SVR on nonlinear data', fontsize=14)
plt.tight_layout()
plt.show()

---
## 6. Tuning SVR: C, Epsilon, Kernel, Gamma

The four main hyperparameters interact with each other:

| Parameter | Effect when increased |
|-----------|-----------------------|
| **C** | Less regularisation; model tries harder to fit every point |
| **epsilon** | Wider tube; more points ignored; smoother/simpler model |
| **gamma** (RBF) | Narrower Gaussian bumps; more complex, localised fit |
| **degree** (poly) | Higher-degree polynomial; more flexible but risk of oscillation |

We use `GridSearchCV` to search the joint space.

In [ ]:
pipe_tune = Pipeline([
    ('scaler', StandardScaler()),
    ('svr', SVR())
])

param_grid = {
    'svr__kernel': ['rbf'],
    'svr__C': [0.1, 1, 10, 100, 1000],
    'svr__epsilon': [0.01, 0.05, 0.1, 0.5, 1.0],
    'svr__gamma': ['scale', 0.01, 0.1, 1],
}

grid = GridSearchCV(pipe_tune, param_grid, cv=5,
                    scoring='neg_mean_squared_error',
                    return_train_score=True, n_jobs=-1)
grid.fit(X_tr_nl, y_tr_nl)

print(f"Best parameters:  {grid.best_params_}")
print(f"Best CV RMSE:     {np.sqrt(-grid.best_score_):.4f}")
print(f"Test RMSE:        {np.sqrt(mean_squared_error(y_te_nl, grid.predict(X_te_nl))):.4f}")
print(f"Test R2:          {r2_score(y_te_nl, grid.predict(X_te_nl)):.4f}")

In [ ]:
# Visualise the tuned model
y_tuned = grid.predict(X_nl_plot)

best_eps = grid.best_params_['svr__epsilon']

fig, ax = plt.subplots(figsize=(10, 6))
ax.fill_between(X_nl_plot.ravel(), y_tuned - best_eps, y_tuned + best_eps,
                alpha=0.2, color='green', label=f'$\\varepsilon$-tube ($\\varepsilon$={best_eps})')
ax.plot(X_nl_plot, y_tuned, 'r-', linewidth=2, label='Tuned SVR')
ax.scatter(X_tr_nl, y_tr_nl, c='steelblue', edgecolors='k', s=50, label='Train')
ax.scatter(X_te_nl, y_te_nl, c='orange', edgecolors='k', s=50, label='Test')
ax.set_title(f'Tuned SVR — C={grid.best_params_["svr__C"]}, '
             f'$\\varepsilon$={best_eps}, '
             f'gamma={grid.best_params_["svr__gamma"]}')
ax.set_xlabel('x')
ax.set_ylabel('y')
ax.legend(loc='best')
plt.tight_layout()
plt.show()

---
## 7. Visualising the Epsilon Tube in Detail

Let us build a more detailed visualisation that clearly marks which points are inside
the tube (zero loss) and which are outside (non-zero loss), and draws residual lines.

In [ ]:
# Fit a clean SVR for visualisation
eps_vis = 0.25
svr_vis = SVR(kernel='rbf', C=10, epsilon=eps_vis, gamma='scale')
svr_vis.fit(X_demo, y_demo)

y_vis_pred = svr_vis.predict(X_demo)
residuals = np.abs(y_demo - y_vis_pred)
inside = residuals <= eps_vis
outside = ~inside

y_vis_line = svr_vis.predict(X_plot)

fig, ax = plt.subplots(figsize=(12, 7))

# Tube
ax.fill_between(X_plot.ravel(), y_vis_line - eps_vis, y_vis_line + eps_vis,
                alpha=0.15, color='green', label=f'$\\varepsilon$-tube ($\\varepsilon$={eps_vis})')
ax.plot(X_plot, y_vis_line, 'r-', linewidth=2, label='SVR fit')
ax.plot(X_plot, y_vis_line + eps_vis, 'g--', linewidth=1, alpha=0.7)
ax.plot(X_plot, y_vis_line - eps_vis, 'g--', linewidth=1, alpha=0.7)

# Points inside tube (zero loss)
ax.scatter(X_demo[inside], y_demo[inside], c='steelblue', edgecolors='k',
           s=50, zorder=3, label=f'Inside tube ({inside.sum()} pts — zero loss)')

# Points outside tube (non-zero loss)
ax.scatter(X_demo[outside], y_demo[outside], c='tomato', edgecolors='k',
           s=60, zorder=3, marker='D',
           label=f'Outside tube ({outside.sum()} pts — penalised)')

# Draw residual lines for outside points
for idx in np.where(outside)[0]:
    xi, yi = X_demo[idx, 0], y_demo[idx]
    yi_hat = y_vis_pred[idx]
    tube_edge = yi_hat + eps_vis if yi > yi_hat else yi_hat - eps_vis
    ax.plot([xi, xi], [tube_edge, yi], 'tomato', linewidth=1.5, alpha=0.7)

ax.set_xlabel('x', fontsize=12)
ax.set_ylabel('y', fontsize=12)
ax.set_title('SVR epsilon tube: inside points (zero loss) vs outside points (penalised)',
             fontsize=13)
ax.legend(loc='best', fontsize=10)
plt.tight_layout()
plt.show()

The red diamond-shaped points lie outside the tube and their residual lines (beyond the
tube edge) represent the slack — the portion of the error that actually contributes to
the loss. All blue circles sit comfortably inside the tube and incur zero penalty.

---
## 8. Effect of C on SVR

Like in classification, C balances the flatness of the model against data fidelity.
With SVR:
- **Small C**: the model prioritises a flat (simple) function, even if many points
  fall outside the tube.
- **Large C**: the model tries to get every point inside (or close to) the tube,
  potentially overfitting.

In [ ]:
C_values_svr = [0.01, 0.1, 1, 10, 100, 1000]

fig, axes = plt.subplots(2, 3, figsize=(18, 10))

for ax, C_val in zip(axes.ravel(), C_values_svr):
    svr_c = SVR(kernel='rbf', C=C_val, epsilon=0.15, gamma='scale')
    svr_c.fit(X_demo, y_demo)
    y_c = svr_c.predict(X_plot)

    ax.fill_between(X_plot.ravel(), y_c - 0.15, y_c + 0.15,
                    alpha=0.15, color='green')
    ax.plot(X_plot, y_c, 'r-', linewidth=2)
    ax.scatter(X_demo, y_demo, c='steelblue', edgecolors='k', s=40)
    ax.scatter(X_demo[svr_c.support_], y_demo[svr_c.support_],
               s=120, facecolors='none', edgecolors='gold', linewidths=2)
    r2 = r2_score(y_demo, svr_c.predict(X_demo))
    ax.set_title(f'C = {C_val}\n$R^2$ = {r2:.3f}, SVs = {len(svr_c.support_)}')
    ax.set_xlabel('x')
    ax.set_ylabel('y')
    ax.set_ylim(-1.8, 1.8)

plt.suptitle('Effect of C on SVR (RBF kernel)', fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

---
## 9. Gamma Effect on SVR

Gamma in the RBF kernel controls locality, the same way as in classification. Let us
see its effect on the regression curve.

In [ ]:
gamma_values_svr = [0.01, 0.1, 1, 5, 20]

fig, axes = plt.subplots(1, 5, figsize=(25, 4.5))

for ax, g in zip(axes, gamma_values_svr):
    svr_g = SVR(kernel='rbf', C=10, epsilon=0.1, gamma=g)
    svr_g.fit(X_demo, y_demo)
    y_g = svr_g.predict(X_plot)

    ax.plot(X_plot, y_g, 'r-', linewidth=2)
    ax.scatter(X_demo, y_demo, c='steelblue', edgecolors='k', s=40)
    r2 = r2_score(y_demo, svr_g.predict(X_demo))
    ax.set_title(f'gamma = {g}\n$R^2$ = {r2:.3f}')
    ax.set_xlabel('x')
    ax.set_ylabel('y')
    ax.set_ylim(-1.8, 1.8)

plt.suptitle('Effect of gamma on SVR (RBF kernel)', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

Small gamma produces an almost flat line (underfitting). Large gamma makes the curve
wrap tightly around each data point (overfitting). A moderate gamma captures the
underlying sine-like pattern without memorising noise.

---
## 10. Pros and Cons of SVM (Classification + Regression)

### Pros

1. **Effective in high-dimensional spaces** — works well even when features > samples.
2. **Memory-efficient** — only support vectors are stored for prediction.
3. **Flexible** — the kernel trick allows modelling complex nonlinear relationships
   without explicit feature engineering.
4. **Robust to overfitting** (in high dimensions) — the margin maximisation principle
   provides built-in regularisation.
5. **Well-suited for small-to-medium datasets** with clear margin of separation.

### Cons

1. **Slow on large datasets** — training time scales roughly $O(n^2)$ to $O(n^3)$.
   For > 100k samples, consider `LinearSVC`/`LinearSVR` or SGD-based alternatives.
2. **Sensitive to feature scaling** — always standardise or normalise features.
3. **Hyperparameter tuning is essential** — C, gamma, epsilon, and kernel all interact;
   grid/random search is typically required.
4. **No built-in feature importance** — unlike tree-based models, SVMs do not directly
   tell you which features matter most.
5. **Probability estimates are expensive** — Platt scaling (`probability=True`) adds
   computational cost and may not be well-calibrated.
6. **Poor with very noisy, overlapping classes** — the margin concept loses its power
   when classes are heavily intermixed.

---
## 11. Summary: When to Use SVM vs Other Methods

| Scenario | Recommended method | Why |
|----------|-------------------|-----|
| Small-to-medium dataset, clear class separation | **SVM (RBF)** | Maximum-margin principle shines |
| High-dimensional sparse data (e.g. text) | **Linear SVM** | Fast, effective, scales well |
| Very large dataset (>100k samples) | **SGDClassifier / Random Forest / Gradient Boosting** | SVM training is too slow |
| Need probability outputs | **Logistic Regression / Random Forest** | SVM probabilities are expensive and approximate |
| Need feature importance | **Tree-based methods** | SVM is a black box |
| Nonlinear regression, small dataset | **SVR (RBF)** | Flexible, robust with proper tuning |
| Large-scale regression | **Linear regression / Ridge / Gradient Boosting** | SVR does not scale |
| Noisy data, many outliers | **Robust regression / Random Forest** | SVM margin is distorted by noise |

**Rule of thumb**: SVM is an excellent *second model* to try. Start with a simple
baseline (logistic regression / linear regression), then try SVM if you suspect
nonlinear patterns and your dataset is not too large. Always use a `Pipeline` with
`StandardScaler` and tune hyperparameters with `GridSearchCV`.

In [ ]:
# Final comparison: SVR vs Linear Regression on a larger nonlinear problem
np.random.seed(0)
n = 200
X_final = np.sort(np.random.uniform(0, 10, n)).reshape(-1, 1)
y_final = (np.sin(X_final) + 0.3 * np.cos(3 * X_final)).ravel() + np.random.normal(0, 0.3, n)

X_tr_f, X_te_f, y_tr_f, y_te_f = train_test_split(
    X_final, y_final, test_size=0.25, random_state=42
)

models = [
    ('Linear Regression', Pipeline([
        ('scaler', StandardScaler()),
        ('lr', LinearRegression())
    ])),
    ('SVR (linear)', Pipeline([
        ('scaler', StandardScaler()),
        ('svr', SVR(kernel='linear', C=10, epsilon=0.1))
    ])),
    ('SVR (RBF)', Pipeline([
        ('scaler', StandardScaler()),
        ('svr', SVR(kernel='rbf', C=100, epsilon=0.1, gamma='scale'))
    ])),
]

X_line_f = np.linspace(0, 10, 400).reshape(-1, 1)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for ax, (name, model) in zip(axes, models):
    model.fit(X_tr_f, y_tr_f)
    y_line_f = model.predict(X_line_f)

    ax.scatter(X_tr_f, y_tr_f, c='steelblue', edgecolors='k', s=30, alpha=0.5, label='Train')
    ax.scatter(X_te_f, y_te_f, c='orange', edgecolors='k', s=40, label='Test')
    ax.plot(X_line_f, y_line_f, 'r-', linewidth=2, label='Prediction')

    test_r2 = r2_score(y_te_f, model.predict(X_te_f))
    test_rmse = np.sqrt(mean_squared_error(y_te_f, model.predict(X_te_f)))
    ax.set_title(f'{name}\nTest $R^2$={test_r2:.3f}  RMSE={test_rmse:.3f}')
    ax.set_xlabel('x')
    ax.set_ylabel('y')
    ax.legend(loc='best')

plt.suptitle('Model comparison on nonlinear regression', fontsize=14)
plt.tight_layout()
plt.show()

---

**Key takeaways:**

1. SVR uses an epsilon-insensitive tube — points inside the tube incur zero loss.
2. C, epsilon, and gamma (for RBF) all interact and must be tuned together.
3. The kernel trick gives SVR the power to fit nonlinear relationships without
   explicit feature engineering.
4. Always scale features and use a Pipeline.
5. SVMs are best for small-to-medium datasets; for large-scale problems, consider
   linear models or tree-based ensembles.